In [7]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration



In [8]:
train_data = pd.read_csv('samsum-train.csv')
val_data = pd.read_csv('samsum-validation.csv')

In [9]:
#random sampling
train_data = train_data.sample(n = 4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n = 500, random_state=42).reset_index(drop=True)

In [10]:
# Data pre-processing
import re
def clean_text(text):
  text = re.sub(r'\r\n', '', text)       #lines
  text = re.sub(r'\s+', '', text)    #spaces
  text = re.sub(r'<.*?>', '', text)   # html tags
  text = text.strip().lower()
  return text

In [11]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_text)
train_data['summary'] = train_data['summary'].apply(clean_text)

val_data['dialogue'] = val_data['dialogue'].apply(clean_text)
val_data['summary'] = val_data['summary'].apply(clean_text)

In [12]:
# data tokenzing
tokenizer = T5Tokenizer.from_pretrained('t5-small')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [13]:
# creating function that convert raw data into tokenized inputs for fine-tune
def tokenize(data):
  inputs = tokenizer(data['dialogue'], padding = 'max_length', max_length=512, truncation=True)
  targets = tokenizer(data['summary'], padding = 'max_length', max_length=150, truncation=True)

  inputs['labels'] = targets['input_ids']
  return inputs

In [14]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [15]:
# working with our model

model = T5ForConditionalGeneration.from_pretrained('t5-small')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [16]:
import torch
if torch.backends.mps.is_available():
  device = torch.device('mps')
elif torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')

print('device :', device)

model.to(device)

device : mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [17]:
#  training arguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    warmup_steps=500
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,

)

# Start training
trainer.train()

/opt/homebrew/anaconda3/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.584295,0.697826
2,0.710873,0.624368
3,0.657993,0.605800
4,0.636029,0.592574
5,0.622419,0.583187
6,0.616093,0.581964


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/homebrew/anaconda3/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/opt/homebrew/anaconda3/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/homebrew/anaconda3/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=1.1379502665201824, metrics={'train_runtime': 2984.3858, 'train_samples_per_second': 8.042, 'train_steps_per_second': 1.005, 'total_flos': 3248203235328000.0, 'train_loss': 1.1379502665201824, 'epoch': 6.0})

In [19]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [20]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [21]:
# Testing the core logic for summarization
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)

  # tokenize
  inputs = tokenizer(dialogue, padding='max_length', max_length=512, truncation=True, return_tensors='pt').to(device)


  # generating the summary
  targets = model.generate(
      input_ids = inputs['input_ids'],
      attention_mask = inputs['attention_mask'],
      max_length = 150,
      num_beams = 4,
      early_stopping=True
  )

  # token ids converted to summary
  summary = tokenize.decode(targets[0], skip_special_tokens=True)
  return summary